In [88]:
import pandas as pd
import torch 
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from torch.utils.data import Subset

In [89]:
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

In [90]:
print(train_data.shape)
print(test_data.shape)
headers = train_data.columns.tolist
display(train_data.head())

(1460, 81)
(1459, 80)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [91]:
train_features = train_data.drop(columns=["Id", "SalePrice"])
test_features = test_data.drop(columns=["Id"])

# 各种数据类型分别有多少列
print(train_features.dtypes.value_counts())

print("*"*10)


# 看看类别型特征的实际取值
display(train_features.select_dtypes(include="object").head(4).T.head(20))

print("*"*10)

# 哪些特征缺失值最多？
display(train_features.isna().sum().sort_values(ascending=False).head(15))

str        43
int64      33
float64     3
Name: count, dtype: int64
**********


C:\Users\lishi\AppData\Local\Temp\ipykernel_58600\673631940.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(train_features.select_dtypes(include="object").head(4).T.head(20))


,0,1,2,3
MSZoning,RL,RL,RL,RL
Street,Pave,Pave,Pave,Pave
Alley,NaN,NaN,NaN,NaN
LotShape,Reg,Reg,IR1,IR1
LandContour,Lvl,Lvl,Lvl,Lvl
Utilities,AllPub,AllPub,AllPub,AllPub
LotConfig,Inside,FR2,Inside,Corner
LandSlope,Gtl,Gtl,Gtl,Gtl
Neighborhood,CollgCr,Veenker,CollgCr,Crawfor
Condition1,Norm,Feedr,Norm,Norm


**********


PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageQual        81
GarageFinish      81
GarageType        81
GarageYrBlt       81
GarageCond        81
BsmtFinType2      38
BsmtExposure      38
BsmtQual          37
dtype: int64

In [92]:
# 合并train和test方便数据处理
all_features = pd.concat(
    [train_features,test_features],
    axis=0,
    ignore_index=True
)

print(all_features.shape)

(2919, 79)


In [93]:
# 处理数字类列
numeric_cols = all_features.select_dtypes(include="number").columns
mean = all_features[numeric_cols].mean()
std = all_features[numeric_cols].std()
all_features[numeric_cols] = (all_features[numeric_cols] - mean) / std.replace(0,1)
all_features[numeric_cols] = all_features[numeric_cols].fillna(0)


In [94]:
# 处理判断类列
all_features = pd.get_dummies(all_features,dummy_na=True,dtype=float)

print(all_features.shape)


(2919, 330)


In [95]:
#分割train&test
train_x = all_features.iloc[:1460]
test_x = all_features.iloc[1460:]
display(train_x.shape,test_x.shape)

(1460, 330)

(1459, 330)

In [96]:
#转为tensor
train_x = torch.tensor(train_x.to_numpy(), dtype=torch.float32)
test_x = torch.tensor(test_x.to_numpy(), dtype=torch.float32)

train_y = torch.tensor(
    train_data["SalePrice"].to_numpy(),
    dtype=torch.float32
).reshape(-1, 1)

print(train_x.shape)
print(test_x.shape)
print(train_y.shape)

torch.Size([1460, 330])
torch.Size([1459, 330])
torch.Size([1460, 1])


In [97]:
#load data
# dataset = TensorDataset(train_x, train_y)

# train_iter = DataLoader(
#     dataset,
#     batch_size=100,
#     shuffle=True
# )

In [106]:
# 一层linear-kfold交叉验证(调超参)

# kfold返回索引
kf = KFold(         
    n_splits=5,
    shuffle = True,
    random_state=42
)
#获得数据
num_epochs = 100
sum_logrmse = 0
loss = nn.MSELoss()
dataset = TensorDataset(train_x, train_y)
for train_idx , val_idx in kf.split(train_x):
    #subset按照指引的索引从已有的数据集中挑选一部分样本
    train_dataset = Subset(dataset,train_idx.tolist())
    val_dataset = Subset(dataset,val_idx.tolist())
    
    train_iter = DataLoader(
        train_dataset,batch_size=100,shuffle=True
    )
    val_iter = DataLoader(
        val_dataset,batch_size=100,shuffle=False
    )
    net = nn.Linear(train_x.shape[1],1)
    optimizer = torch.optim.Adam(net.parameters(),lr=0.01)
    
    for epoch in range(num_epochs):
        for x,y in train_iter:
            optimizer.zero_grad()
            y_hat = net(x)
            l = loss(torch.log(y),y_hat)
            l.backward()
            optimizer.step()
            
    net.eval()
    log_sum = 0
    total = 0
    with torch.no_grad():
        for x,y in val_iter:
            y_hat = net(x)
            log_erorr = y_hat-torch.log(y)
            log_sum += (log_erorr**2).sum().item()
            total += y.shape[0]
            
    logrmse = (log_sum / total)**0.5
    sum_logrmse += logrmse
    

cvlogrmse = 0.2*sum_logrmse

display(cvlogrmse)


        


0.1522610783097023

In [101]:
#交叉验证之后使用所有数据重练习
full_dataset = TensorDataset(train_x,train_y)

full_iter = DataLoader(
    full_dataset,
    batch_size=100,
    shuffle=True
)

net = nn.Linear(train_x.shape[1],1)
optimizer = torch.optim.Adam(net.parameters(),lr=0.01)
loss = nn.MSELoss()
for epoch in range(100):
    for x,y in full_iter:
        optimizer.zero_grad()
        y_hat = net(x)
        l = loss(torch.log(y),y_hat)
        l.backward()
        optimizer.step()
        
net.eval()

with torch.no_grad():
    predictions = torch.exp(net(test_x)).squeeze(1).cpu().numpy()
    

In [105]:
submission = pd.DataFrame({
    "Id": test_data["Id"],
    "SalePrice": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

display(submission.head())

,Id,SalePrice
0,1461,120423.914062
1,1462,146200.812500
2,1463,180147.468750
3,1464,199292.328125
4,1465,195883.250000


In [ ]:
#xgboost vs residual mlp
import copy
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold

try:
    from xgboost import XGBRegressor
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBRegressor

# ========= 1. 准备数据 =========

def as_numpy(a):
    if isinstance(a, torch.Tensor):
        return a.detach().cpu().numpy().astype(np.float32)
    if hasattr(a, "to_numpy"):
        return a.to_numpy(dtype=np.float32)
    return np.asarray(a, dtype=np.float32)

X = np.ascontiguousarray(as_numpy(train_x))
X_test = np.ascontiguousarray(as_numpy(test_x))

# 始终从原始 SalePrice 读取，避免重复取 log
y_log = np.log1p(train_data["SalePrice"].to_numpy(dtype=np.float64))

assert X.shape[0] == len(y_log)
assert X.shape[1] == X_test.shape[1]
assert np.isfinite(X).all() and np.isfinite(X_test).all()

# 避免小模型使用过多 CPU 线程
torch.set_num_threads(min(4, torch.get_num_threads()))


# ========= 2. 深度学习：Residual MLP =========

class ResMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.GELU()
        )

        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.LayerNorm(128),
                nn.Linear(128, 128),
                nn.GELU(),
                nn.Dropout(0.10),
                nn.Linear(128, 128)
            )
            for _ in range(2)
        ])

        self.head = nn.Sequential(
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        h = self.stem(x)
        for block in self.blocks:
            h = h + block(h)  # 残差连接
        return self.head(h).squeeze(1)


def fit_mlp(X_tr, y_tr, X_va, y_va, X_te, seed):
    torch.manual_seed(seed)

    # 对 log 房价再次标准化，帮助优化器训练
    mu = y_tr.mean()
    sigma = y_tr.std()

    dataset = TensorDataset(
        torch.from_numpy(X_tr),
        torch.from_numpy(
            ((y_tr - mu) / sigma).astype(np.float32)
        )
    )
    loader = DataLoader(
        dataset, batch_size=128, shuffle=True, num_workers=0
    )

    model = ResMLP(X_tr.shape[1])
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=0.001, weight_decay=0.001
    )
    loss_fn = nn.MSELoss()

    best_error = float("inf")
    best_state = None
    patience = 30
    wait = 0

    for epoch in range(200):
        model.train()

        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()

        # Early Stopping：观察验证集误差
        model.eval()
        with torch.no_grad():
            pred = model(torch.from_numpy(X_va)).numpy()
            pred = pred * sigma + mu

        error = np.sqrt(np.mean((pred - y_va) ** 2))

        if error < best_error - 1e-5:
            best_error = error
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    # 恢复验证误差最低时的权重
    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_val = model(torch.from_numpy(X_va)).numpy() * sigma + mu
        pred_test = model(torch.from_numpy(X_te)).numpy() * sigma + mu

    return pred_val, pred_test, epoch + 1


# ========= 3. 五折对照实验 =========

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_mlp = np.zeros(len(X))
oof_xgb = np.zeros(len(X))

test_log_mlp = np.zeros(len(X_test))
test_log_xgb = np.zeros(len(X_test))

for fold, (tr, va) in enumerate(kf.split(X), start=1):

    # --- 模型 A：Residual MLP ---
    p_val, p_test, epochs = fit_mlp(
        X[tr], y_log[tr],
        X[va], y_log[va],
        X_test,
        seed=2026 + fold
    )

    oof_mlp[va] = p_val
    test_log_mlp += p_test / 5

    # --- 模型 B：XGBoost ---
    xgb = XGBRegressor(
        n_estimators=2500,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=2,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5,
        objective="reg:squarederror",
        tree_method="hist",
        eval_metric="rmse",
        early_stopping_rounds=80,
        n_jobs=-1,
        random_state=2026 + fold
    )

    xgb.fit(
        X[tr], y_log[tr],
        eval_set=[(X[va], y_log[va])],
        verbose=False
    )

    oof_xgb[va] = xgb.predict(X[va])
    test_log_xgb += xgb.predict(X_test) / 5

    mlp_score = np.sqrt(np.mean((oof_mlp[va] - y_log[va]) ** 2))
    xgb_score = np.sqrt(np.mean((oof_xgb[va] - y_log[va]) ** 2))

    print(
        f"Fold {fold}: "
        f"MLP={mlp_score:.5f}, "
        f"XGB={xgb_score:.5f}, "
        f"MLP epochs={epochs}, "
        f"XGB trees={xgb.best_iteration + 1}"
    )


# ========= 4. 比较交叉验证成绩 =========

def cv_score(pred):
    return np.sqrt(np.mean((pred - y_log) ** 2))

print("\n===== 5-Fold OOF Results =====")
print(f"Residual MLP : {cv_score(oof_mlp):.5f}")
print(f"XGBoost      : {cv_score(oof_xgb):.5f}")

# 额外试一个模型融合：对数预测加权平均
blend_oof = 0.3 * oof_mlp + 0.7 * oof_xgb
print(f"Blend        : {cv_score(blend_oof):.5f}")


# ========= 5. 生成三个 Kaggle 提交文件 =========

outputs = {
    "submission_mlp.csv": test_log_mlp,
    "submission_xgb.csv": test_log_xgb,
    "submission_blend.csv": (
        0.3 * test_log_mlp + 0.7 * test_log_xgb
    )
}

for filename, prediction_log in outputs.items():
    submission = pd.DataFrame({
        "Id": test_data["Id"].to_numpy(),
        "SalePrice": np.expm1(np.maximum(prediction_log, 0))
    })

    submission.to_csv(filename, index=False)
    print("Saved:", os.path.abspath(filename), submission.shape)


Fold 1: MLP=0.13627, XGB=0.13040, MLP epochs=41, XGB trees=841
Fold 2: MLP=0.12142, XGB=0.11204, MLP epochs=35, XGB trees=740
Fold 3: MLP=0.14877, XGB=0.16200, MLP epochs=38, XGB trees=195
Fold 4: MLP=0.12958, XGB=0.12405, MLP epochs=40, XGB trees=1111
Fold 5: MLP=0.10669, XGB=0.10372, MLP epochs=46, XGB trees=961

===== 5-Fold OOF Results =====
Residual MLP : 0.12932
XGBoost      : 0.12802
Blend        : 0.12282
Saved: c:\Users\lishi\Desktop\python\pytorch\house_price\submission_mlp.csv (1459, 2)
Saved: c:\Users\lishi\Desktop\python\pytorch\house_price\submission_xgb.csv (1459, 2)
Saved: c:\Users\lishi\Desktop\python\pytorch\house_price\submission_blend.csv (1459, 2)
